In [2]:
"""
fetch_football_data.py
======================
Downloads historical football match data (2000/01 → 2023/24) from
football-data.co.uk and converts it to the exact same schema as
football_matches_2024_2025.csv — including consistent IDs.

ID STRATEGY
-----------
  match_id     : sequential, starting at max(2024/25 match_id) + 1
  home/away_team_id : reused from 2024/25 where team name matches;
                      new teams get IDs starting above max(team_id)
  referee_id   : reused from 2024/25 where name matches;
                      new referees get IDs starting above max(referee_id)
  matchday     : set to 1 (not available from this source)

COVERAGE
--------
  PL  — Premier League   FL1 — Ligue 1
  BL1 — Bundesliga       SA  — Serie A
  PD  — La Liga
  (Champions League not available on this free source)

USAGE
-----
  pip install requests pandas
  python fetch_football_data.py
"""

import requests
import pandas as pd
import io
import os
import time

# ── CONFIG ────────────────────────────────────────────────────────────────────

BASE_URL        = "https://www.football-data.co.uk/mmz4281"
OUTPUT_DIR      = "football_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

COMPETITIONS = {
    "PL":  ("E0",  "Premier League"),
    "BL1": ("D1",  "Bundesliga"),
    "SA":  ("I1",  "Serie A"),
    "PD":  ("SP1", "La Liga"),
    "FL1": ("F1",  "Ligue 1"),
}

SEASONS = list(range(2000, 2024))   # 2000/01 → 2023/24


# ── ID REGISTRIES (mutable, shared across all seasons) ───────────────────────

class IDRegistry:
    def __init__(self, name_to_id: dict, next_id: int):
        self._map     = {k: v for k, v in name_to_id.items()}
        self._next    = next_id

    def get(self, name: str | None) -> int | None:
        if not name or pd.isna(name):
            return None
        name = str(name).strip()
        if not name:
            return None
        if name not in self._map:
            self._map[name] = self._next
            self._next += 1
        return self._map[name]


# ── DOWNLOAD HELPERS ──────────────────────────────────────────────────────────

def season_code(year: int) -> str:
    """2023 → '2324'"""
    return f"{str(year)[-2:]}{str(year + 1)[-2:]}"

def season_label(year: int) -> str:
    """2023 → '2023/24'"""
    return f"{year}/{str(year + 1)[-2:]}"

def fetch_csv(fdco_code: str, year: int) -> pd.DataFrame | None:
    url = f"{BASE_URL}/{season_code(year)}/{fdco_code}.csv"
    try:
        resp = requests.get(url, timeout=20)
        if resp.status_code != 200:
            return None
        df = pd.read_csv(
            io.StringIO(resp.text),
            on_bad_lines="skip",
            encoding="latin-1",
        )
        df = df.dropna(how="all").dropna(axis=1, how="all")
        return df if not df.empty else None
    except Exception:
        return None


# ── ROW CONVERTER ─────────────────────────────────────────────────────────────

def convert_row(
    row: pd.Series,
    comp_code: str,
    comp_name: str,
    year: int,
    match_counter: list,   # [current_id]  — mutable so we can increment
    teams: IDRegistry,
    refs: IDRegistry,
) -> dict | None:

    ft_home_raw = row.get("FTHG")
    ft_away_raw = row.get("FTAG")
    if pd.isna(ft_home_raw) or pd.isna(ft_away_raw):
        return None   # unplayed fixture

    ft_home = int(ft_home_raw)
    ft_away = int(ft_away_raw)
    ht_home_raw = row.get("HTHG")
    ht_away_raw = row.get("HTAG")
    ht_home = int(ht_home_raw) if not pd.isna(ht_home_raw) else None
    ht_away = int(ht_away_raw) if not pd.isna(ht_away_raw) else None

    goal_diff   = ft_home - ft_away
    total_goals = ft_home + ft_away

    if ft_home > ft_away:
        outcome            = "Home Win"
        home_pts, away_pts = 3, 0
    elif ft_home < ft_away:
        outcome            = "Away Win"
        home_pts, away_pts = 0, 3
    else:
        outcome            = "Draw"
        home_pts, away_pts = 1, 1

    # Normalise date
    raw_date = str(row.get("Date", "")).strip()
    try:
        date_utc = pd.to_datetime(raw_date, dayfirst=True).strftime("%Y-%m-%d")
    except Exception:
        date_utc = raw_date

    home_name = str(row.get("HomeTeam", "")).strip()
    away_name = str(row.get("AwayTeam", "")).strip()
    ref_name  = str(row.get("Referee",  "")).strip() if "Referee" in row.index else None

    # Assign match_id
    mid = match_counter[0]
    match_counter[0] += 1

    return {
        "competition_code": comp_code,
        "competition_name": comp_name,
        "season":           season_label(year),
        "match_id":         mid,
        "matchday":         1,
        "stage":            "REGULAR_SEASON",
        "status":           "FINISHED",
        "date_utc":         date_utc,
        "referee":          ref_name if ref_name else None,
        "home_team_id":     teams.get(home_name),
        "home_team":        home_name,
        "away_team_id":     teams.get(away_name),
        "away_team":        away_name,
        "fulltime_home":    ft_home,
        "fulltime_away":    ft_away,
        "halftime_home":    ht_home,
        "halftime_away":    ht_away,
        "goal_difference":  goal_diff,
        "total_goals":      total_goals,
        "match_outcome":    outcome,
        "home_points":      home_pts,
        "away_points":      away_pts,
        "referee_id":       refs.get(ref_name),
    }


# ── MAIN ──────────────────────────────────────────────────────────────────────

def main():
    print("=" * 62)
    print("  football-data.co.uk fetcher — with consistent IDs")
    print("=" * 62)

    teams         = IDRegistry({}, 1)
    refs          = IDRegistry({}, 1)
    match_counter = [1]

    all_frames = []

    for year in SEASONS:
        lbl = season_label(year)
        print(f"\n  {lbl}")
        season_rows = []

        for comp_code, (fdco_code, comp_name) in COMPETITIONS.items():
            print(f"  ↳ {comp_name:<35}", end=" ", flush=True)

            df_raw = fetch_csv(fdco_code, year)
            if df_raw is None:
                print("not found")
                continue

            parsed = [
                r for _, row in df_raw.iterrows()
                if (r := convert_row(row, comp_code, comp_name, year,
                                     match_counter, teams, refs))
            ]
            season_rows.extend(parsed)
            print(f"{len(parsed)} matches")
            time.sleep(0.3)

        if season_rows:
            df    = pd.DataFrame(season_rows)
            fname = f"football_matches_{year}_{year + 1}.csv"
            fpath = os.path.join(OUTPUT_DIR, fname)
            df.to_csv(fpath, index=False)
            print(f"  {len(df):,} rows → {fpath}")
            all_frames.append(df)

    # Combined file
    if all_frames:
        combined  = pd.concat(all_frames, ignore_index=True)
        out_path  = os.path.join(OUTPUT_DIR, "football_matches_ALL.csv")
        combined.to_csv(out_path, index=False)

        print("\n" + "=" * 62)
        print("  Done!")
        print(f"    Combined file  : {out_path}")
        print(f"    Total rows     : {len(combined):,}")
        print(f"    Seasons        : {combined['season'].nunique()}")
        print(f"    Date range     : {combined['date_utc'].min()} → {combined['date_utc'].max()}")
        print(f"    Unique teams   : {pd.concat([combined['home_team'], combined['away_team']]).nunique()}")
        print(f"    Unique referees: {combined['referee'].nunique()}")
        print("=" * 62)


if __name__ == "__main__":
    main()

  football-data.co.uk fetcher — with consistent IDs

  2000/01
  ↳ Premier League                      380 matches
  ↳ Bundesliga                          306 matches
  ↳ Serie A                             306 matches
  ↳ La Liga                             380 matches
  ↳ Ligue 1                             306 matches
  1,678 rows → football_data/football_matches_2000_2001.csv

  2001/02
  ↳ Premier League                      380 matches
  ↳ Bundesliga                          306 matches
  ↳ Serie A                             306 matches
  ↳ La Liga                             380 matches
  ↳ Ligue 1                             306 matches
  1,678 rows → football_data/football_matches_2001_2002.csv

  2002/03
  ↳ Premier League                      380 matches
  ↳ Bundesliga                          290 matches
  ↳ Serie A                             306 matches
  ↳ La Liga                             380 matches
  ↳ Ligue 1                             191 matches
  1,547 rows → 